In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data

In [19]:
## Parameters for encoder/decoder blocks and layers

d_model = 512 # dimension of inputs to enc/dec layers
n_heads = 8 # number of attention heads
n_layers = 6 # number of enc/dec layers
latent_dim = 2 # dimension of latent variable space

In [ ]:
# How to use nested tensors and get rid of warning
class VAETransformer(nn.Module):
    def __init__(self, dmodel, nheads, nlayers, latent_dim):
        super().__init__()
        # need tokenization, embeddings
        encoder_layer = nn.TransformerEncoderLayer(d_model=dmodel, 
                                                   nhead = nheads)
        self.encoder =  nn.TransformerEncoder(encoder_layer, nlayers)
        decoder_layer = nn.TransformerDecoderLayer(d_model=dmodel, 
                                                   nhead=nheads)
        self.decoder = nn.TransformerDecoder(decoder_layer, nlayers)

        self.mean_layer = nn.Linear(dmodel, latent_dim)
        self.logvar_layer = nn.Linear (dmodel, latent_dim)

        self.latent_to_decode = nn.Linear(latent_dim, dmodel)

    def forward(self, x):
        memory = self.encoder(x)
        mean = self.mean_layer(memory) 
        # do we want (seq_len, batch, latent_dim)
        # or do we want (1, batch, latent_dim) so mapping sequences to latent varrs
        log_var = self.logvar_layer(memory)
        z = self.reparameterization(mean, torch.exp(0.5 * log_var))
        x = self.latent_to_decode(z)
        x = self.decoder(x, memory) # what should the memory be?
        return x, mean, log_var

    def reparameterization(self, mean, var):
        epsilon = torch.randn_like(var)
        z = mean + var*epsilon
        return z

In [23]:
vae_transformer = VAETransformer(d_model, n_heads, n_layers, latent_dim) 

/scratch/404052.1.ood/ipykernel_725746/899198568.py:7: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder =  nn.TransformerEncoder(encoder_layer, nlayers)


In [29]:

src = torch.rand(10, 32, 512)
x, mean, logvar = vae_transformer(src)
print(x.shape)
print(mean.shape)

torch.Size([10, 32, 512])
torch.Size([10, 32, 2])
